### Import a CSV dataset

You can use the `pandas` library to easily import your CSV file into a DataFrame. Make sure the CSV file is in the same directory as your notebook or provide the full path to the file.

In [4]:
import pandas as pd

df = pd.read_csv(
    "/content/BTC.tsv.tsv",
    sep="\t",
    encoding="utf-16"
)

df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/BTC.tsv.tsv'

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Renombrar columnas
df = df.rename(columns={
    "Time": "date",
    "BTC / USD Denominated Closing Price": "price"
})

# Convertir tipos
df["date"] = pd.to_datetime(df["date"])
df["price"] = pd.to_numeric(df["price"])

# Ordenar por fecha
df = df.sort_values("date")

df.head()

In [ ]:
plt.figure(figsize=(14,6))
plt.plot(df["date"], df["price"])

plt.yscale("log")
plt.title("Bitcoin PriceUSD en escala log")
plt.xlabel("Fecha")
plt.ylabel("Precio BTC/USD log")
plt.grid(True, which="both", alpha=0.3)

plt.show()

In [ ]:
# Log del precio
df["log_price"] = np.log(df["price"])

# Tiempo numérico para poder hacer regresión
df["t"] = (df["date"] - df["date"].min()).dt.days

# Tendencia lineal en log
coef = np.polyfit(df["t"], df["log_price"], 1)
df["trend"] = np.polyval(coef, df["t"])

# Ciclo = desviación respecto a tendencia
df["cycle"] = df["log_price"] - df["trend"]

# Normalizar entre -100 y 100
df["cycle_norm"] = 100 * df["cycle"] / df["cycle"].abs().max()

plt.figure(figsize=(14,6))
plt.plot(df["date"], df["cycle_norm"])
plt.axhline(0, linewidth=2)

plt.title("Bitcoin como oscilador: desviación log normalizada")
plt.xlabel("Fecha")
plt.ylabel("Ciclo normalizado")
plt.grid(True, alpha=0.3)

plt.show()

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator

# =========================
# 1. Preparar dataset
# =========================

df = df.rename(columns={
    "Time": "date",
    "BTC / USD Denominated Closing Price": "price"
})

df["date"] = pd.to_datetime(df["date"])
df["price"] = pd.to_numeric(df["price"])
df = df.sort_values("date")

# Nos quedamos desde 2013
btc = df[df["date"] >= "2013-01-01"].copy()

# Datos semanales: menos ruido que diario, pero mantiene bastante detalle
weekly = (
    btc.set_index("date")
       .resample("W")
       .last()
       .dropna()
       .reset_index()
)

weekly["log_price"] = np.log(weekly["price"])

# =========================
# 2. Detectar máximos/mínimos reales por ventanas
# =========================

def get_turning_point(start, end, kind, label, phase):
    subset = weekly[(weekly["date"] >= start) & (weekly["date"] <= end)].copy()

    if kind == "max":
        row = subset.loc[subset["price"].idxmax()]
    else:
        row = subset.loc[subset["price"].idxmin()]

    return {
        "date": row["date"],
        "price": row["price"],
        "log_price": np.log(row["price"]),
        "label": label,
        "phase": phase
    }

turns = []

# Inicio del ciclo 2013
turns.append(get_turning_point("2013-01-01", "2013-04-30", "min", "Inicio 2013", -0.35))

# Ciclo 2013-2015
turns.append(get_turning_point("2013-10-01", "2014-01-31", "max", "Max 2013", 1.00))
turns.append(get_turning_point("2014-12-01", "2015-03-31", "min", "Min 2015", -0.75))

# Ciclo 2017-2018
turns.append(get_turning_point("2017-11-01", "2018-01-31", "max", "Max 2017", 0.75))
turns.append(get_turning_point("2018-11-01", "2019-02-28", "min", "Min 2018", -0.50))

# Ciclo 2021-2022
turns.append(get_turning_point("2021-10-01", "2021-12-31", "max", "Max 2021", 0.50))
turns.append(get_turning_point("2022-10-01", "2023-01-31", "min", "Min 2022", -0.35))

# Ciclo 2025
turns.append(get_turning_point("2025-01-01", "2026-02-01", "max", "Max 2025", 0.35))

# Proyección suave para poder ubicar el tramo actual dentro del ciclo bajista
# Puedes cambiar estos valores más adelante
projected_min = {
    "date": pd.Timestamp("2026-10-15"),
    "price": 36500,
    "log_price": np.log(36500),
    "label": "Min 2026 proyectado",
    "phase": -0.28
}
turns.append(projected_min)

turns = pd.DataFrame(turns).sort_values("date").reset_index(drop=True)

# =========================
# 3. Normalizar CADA tramo usando datos reales
# =========================
# Esto es lo importante:
# Entre un máximo y un mínimo, usamos todos los precios reales semanales
# y los convertimos a fase del ciclo.

segments = []

for i in range(len(turns) - 1):
    start = turns.iloc[i]
    end = turns.iloc[i + 1]

    seg = weekly[
        (weekly["date"] >= start["date"]) &
        (weekly["date"] <= end["date"])
    ].copy()

    if len(seg) == 0:
        continue

    log_start = start["log_price"]
    log_end = end["log_price"]
    phase_start = start["phase"]
    phase_end = end["phase"]

    # Normalización en log dentro de cada tramo
    # ratio = 0 al inicio del tramo, ratio = 1 al final
    seg["ratio"] = (seg["log_price"] - log_start) / (log_end - log_start)

    # Convertimos ese ratio a fase del oscilador
    seg["cycle_raw"] = phase_start + seg["ratio"] * (phase_end - phase_start)

    # Evitamos que rebotes internos se vayan absurdamente fuera del rango
    low = min(phase_start, phase_end) - 0.12
    high = max(phase_start, phase_end) + 0.12
    seg["cycle_raw"] = seg["cycle_raw"].clip(low, high)

    segments.append(seg)

cycle = pd.concat(segments).sort_values("date").drop_duplicates("date")

# Suavizado ligero: conserva detalle, pero quita ruido semanal feo
cycle["cycle_smooth"] = cycle["cycle_raw"].rolling(5, center=True, min_periods=1).mean()

# =========================
# 4. Gráfico detallado
# =========================

plt.figure(figsize=(16, 7))

plt.plot(
    cycle["date"],
    cycle["cycle_smooth"],
    linewidth=2.5,
    label="Ciclo BTC normalizado con datos reales"
)

plt.plot(
    cycle["date"],
    cycle["cycle_raw"],
    linewidth=0.8,
    alpha=0.35,
    label="Detalle semanal real"
)

plt.axhline(0, linewidth=2.5, label="Tendencia normalizada")

# Puntos clave
plt.scatter(
    turns["date"],
    turns["phase"],
    s=65,
    zorder=5
)

# Etiquetas de puntos clave, pero sin llenar demasiado el gráfico
for _, row in turns.iterrows():
    if "proyectado" in row["label"]:
        continue

    text = f"{row['label']}\n${row['price']:,.0f}"
    offset = 12 if row["phase"] >= 0 else -28
    va = "bottom" if row["phase"] >= 0 else "top"

    plt.annotate(
        text,
        (row["date"], row["phase"]),
        textcoords="offset points",
        xytext=(0, offset),
        ha="center",
        va=va,
        fontsize=8.5
    )

# Punto actual
last = weekly.iloc[-1]
last_cycle = cycle.iloc[-1]

plt.scatter(
    last_cycle["date"],
    last_cycle["cycle_smooth"],
    s=90,
    zorder=6
)

plt.annotate(
    f"Hoy\n${last['price']:,.0f}",
    (last_cycle["date"], last_cycle["cycle_smooth"]),
    textcoords="offset points",
    xytext=(12, -8),
    ha="left",
    va="center",
    fontsize=9
)

# Estilo eje Y
plt.yticks(
    [-1.0, -0.5, 0.0, 0.5, 1.0],
    ["Capitulación", "Infravalorado", "Tendencia", "Sobrecalentado", "Euforia"]
)

plt.ylim(-1.15, 1.15)
plt.title("Bitcoin: oscilador de ciclos desde 2013 con detalle semanal real")
plt.xlabel("Fecha")
plt.ylabel("Fase del ciclo")
plt.grid(True, axis="y", alpha=0.3)
plt.legend()
plt.show()

# Tabla de puntos clave
turns[["date", "label", "price", "phase"]]

NameError: name 'df' is not defined

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 0. Parámetros editables
# ============================================================

# Rango de drawdown esperado desde el máximo 2025 para estimar próximo mínimo
# Ejemplo: -66% a -76%
drawdown_suave = 0.66
drawdown_duro = 0.76

# Multiplicador mínimo → máximo siguiente para estimar próximo máximo
# Ejemplo: x3 a x6 desde el mínimo proyectado
mult_min = 3.0
mult_max = 6.0

# Suavizado semanal del oscilador
rolling_smooth = 1

# ============================================================
# 1. Preparar dataset
# ============================================================

df = df.rename(columns={
    "Time": "date",
    "BTC / USD Denominated Closing Price": "price"
})

df["date"] = pd.to_datetime(df["date"])
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df = df.dropna(subset=["date", "price"])
df = df.sort_values("date")

# Nos quedamos desde 2013
btc = df[df["date"] >= "2013-01-01"].copy()

# Datos semanales para quitar ruido pero mantener detalle
weekly = (
    btc.set_index("date")
       .resample("W")
       .last()
       .dropna()
       .reset_index()
)

weekly["log_price"] = np.log(weekly["price"])

# ============================================================
# 2. Detectar máximos y mínimos reales por ventanas
# ============================================================

def get_turning_point(start, end, kind, label, phase):
    subset = weekly[(weekly["date"] >= start) & (weekly["date"] <= end)].copy()

    if len(subset) == 0:
        raise ValueError(f"No hay datos entre {start} y {end}")

    if kind == "max":
        row = subset.loc[subset["price"].idxmax()]
    else:
        row = subset.loc[subset["price"].idxmin()]

    return {
        "date": row["date"],
        "price": row["price"],
        "log_price": np.log(row["price"]),
        "label": label,
        "phase": phase,
        "kind": kind
    }

turns_actual = []

# Inicio visual 2013
turns_actual.append(get_turning_point("2013-01-01", "2013-04-30", "min", "Inicio 2013", -0.35))

# Ciclo 2013-2015
turns_actual.append(get_turning_point("2013-10-01", "2014-01-31", "max", "Max 2013",  1.00))
turns_actual.append(get_turning_point("2014-12-01", "2015-03-31", "min", "Min 2015", -0.75))

# Ciclo 2017-2018
turns_actual.append(get_turning_point("2017-11-01", "2018-01-31", "max", "Max 2017",  0.75))
turns_actual.append(get_turning_point("2018-11-01", "2019-02-28", "min", "Min 2018", -0.50))

# Ciclo 2021-2022
turns_actual.append(get_turning_point("2021-10-01", "2021-12-31", "max", "Max 2021",  0.50))
turns_actual.append(get_turning_point("2022-10-01", "2023-01-31", "min", "Min 2022", -0.35))

# Ciclo 2025
turns_actual.append(get_turning_point("2025-01-01", "2026-03-01", "max", "Max 2025",  0.35))

turns_actual = pd.DataFrame(turns_actual).sort_values("date").reset_index(drop=True)

# ============================================================
# 3. Calcular amortiguamiento de las envolventes
# ============================================================

def year_diff(d2, d1):
    return (pd.Timestamp(d2) - pd.Timestamp(d1)).days / 365.25

peaks = turns_actual[turns_actual["kind"] == "max"].copy()

troughs = turns_actual[
    (turns_actual["kind"] == "min") &
    (turns_actual["label"] != "Inicio 2013")
].copy()

peak_dates = peaks["date"].to_list()
peak_amps = peaks["phase"].abs().to_numpy()

trough_dates = troughs["date"].to_list()
trough_amps = troughs["phase"].abs().to_numpy()

peak_betas = []

for i in range(len(peak_amps) - 1):
    dt = year_diff(peak_dates[i + 1], peak_dates[i])
    beta = -np.log(peak_amps[i + 1] / peak_amps[i]) / dt
    peak_betas.append(beta)

trough_betas = []

for i in range(len(trough_amps) - 1):
    dt = year_diff(trough_dates[i + 1], trough_dates[i])
    beta = -np.log(trough_amps[i + 1] / trough_amps[i]) / dt
    trough_betas.append(beta)

peak_betas = np.array(peak_betas)
trough_betas = np.array(trough_betas)

beta_peak_mean = peak_betas.mean()
beta_peak_std = peak_betas.std(ddof=1)

beta_trough_mean = trough_betas.mean()
beta_trough_std = trough_betas.std(ddof=1)

# Menos amortiguamiento = envolvente más alta
beta_peak_low = max(0.001, beta_peak_mean - beta_peak_std)

# Más amortiguamiento = envolvente más baja
beta_peak_high = beta_peak_mean + beta_peak_std

beta_trough_low = max(0.001, beta_trough_mean - beta_trough_std)
beta_trough_high = beta_trough_mean + beta_trough_std

# ============================================================
# 4. Estimar ventanas temporales del próximo mínimo y máximo
# ============================================================

peak_to_peak = []

for i in range(len(peak_dates) - 1):
    peak_to_peak.append(year_diff(peak_dates[i + 1], peak_dates[i]))

historical_peak_to_trough = [
    year_diff(troughs.iloc[0]["date"], peaks.iloc[0]["date"]),
    year_diff(troughs.iloc[1]["date"], peaks.iloc[1]["date"]),
    year_diff(troughs.iloc[2]["date"], peaks.iloc[2]["date"])
]

peak_to_peak = np.array(peak_to_peak)
historical_peak_to_trough = np.array(historical_peak_to_trough)

pp_mean = peak_to_peak.mean()
pp_std = peak_to_peak.std(ddof=1)

pt_mean = historical_peak_to_trough.mean()
pt_std = historical_peak_to_trough.std(ddof=1)

last_peak = peaks.iloc[-1]
last_peak_date = last_peak["date"]
last_peak_price = last_peak["price"]

next_min_early = last_peak_date + pd.to_timedelta((pt_mean - pt_std) * 365.25, unit="D")
next_min_late = last_peak_date + pd.to_timedelta((pt_mean + pt_std) * 365.25, unit="D")
next_min_center = last_peak_date + pd.to_timedelta(pt_mean * 365.25, unit="D")

next_max_early = last_peak_date + pd.to_timedelta((pp_mean - pp_std) * 365.25, unit="D")
next_max_late = last_peak_date + pd.to_timedelta((pp_mean + pp_std) * 365.25, unit="D")
next_max_center = last_peak_date + pd.to_timedelta(pp_mean * 365.25, unit="D")

# ============================================================
# 5. Estimar rango de precios próximo mínimo / máximo
# ============================================================

# Mínimo futuro desde drawdown del máximo 2025
min_price_low = last_peak_price * (1 - drawdown_duro)
min_price_high = last_peak_price * (1 - drawdown_suave)

# Máximo futuro desde multiplicador mínimo → máximo
max_price_low = min_price_low * mult_min
max_price_high = min_price_high * mult_max

# ============================================================
# 6. Añadir mínimo futuro oculto para normalizar tramo actual
# ============================================================

last_trough = troughs.iloc[-1]
dt_next_trough = year_diff(next_min_center, last_trough["date"])

next_trough_amp_less_damp = abs(last_trough["phase"]) * np.exp(-beta_trough_low * dt_next_trough)
next_trough_amp_more_damp = abs(last_trough["phase"]) * np.exp(-beta_trough_high * dt_next_trough)

next_trough_phase = -np.mean([
    next_trough_amp_less_damp,
    next_trough_amp_more_damp
])

projected_min = {
    "date": next_min_center,
    "price": (min_price_low + min_price_high) / 2,
    "log_price": np.log((min_price_low + min_price_high) / 2),
    "label": "Min proyectado",
    "phase": next_trough_phase,
    "kind": "min"
}

turns_for_mapping = pd.concat([
    turns_actual,
    pd.DataFrame([projected_min])
]).sort_values("date").reset_index(drop=True)

# ============================================================
# 7. Crear oscilador detallado usando datos reales semanales
# ============================================================

segments = []

for i in range(len(turns_for_mapping) - 1):
    start = turns_for_mapping.iloc[i]
    end = turns_for_mapping.iloc[i + 1]

    # Solo usamos datos reales hasta el último dato disponible
    seg_end_date = min(end["date"], weekly["date"].max())

    seg = weekly[
        (weekly["date"] >= start["date"]) &
        (weekly["date"] <= seg_end_date)
    ].copy()

    if len(seg) == 0:
        continue

    log_start = start["log_price"]
    log_end = end["log_price"]
    phase_start = start["phase"]
    phase_end = end["phase"]

    if abs(log_end - log_start) < 1e-9:
        continue

    # Normalización en log dentro del tramo
    seg["ratio"] = (seg["log_price"] - log_start) / (log_end - log_start)

    seg["cycle_raw"] = phase_start + seg["ratio"] * (phase_end - phase_start)

    # Clipping suave para que rebotes internos no rompan el oscilador
    low = min(phase_start, phase_end) - 0.12
    high = max(phase_start, phase_end) + 0.12

    seg["cycle_raw"] = seg["cycle_raw"].clip(low, high)

    segments.append(seg)

cycle = pd.concat(segments).sort_values("date").drop_duplicates("date")

cycle["cycle_smooth"] = (
    cycle["cycle_raw"]
    .rolling(rolling_smooth, center=True, min_periods=1)
    .mean()
)

# ============================================================
# 8. Crear envolventes exponenciales superior e inferior
# ============================================================

future_end = pd.Timestamp("2031-03-01")

env_dates = pd.date_range(
    peaks.iloc[0]["date"],
    future_end,
    periods=700
)

# Envolvente superior desde Max 2013
upper_anchor_date = peaks.iloc[0]["date"]
upper_anchor_amp = abs(peaks.iloc[0]["phase"])

upper_t = np.array([
    max(0, year_diff(d, upper_anchor_date))
    for d in env_dates
])

upper_less_damp = upper_anchor_amp * np.exp(-beta_peak_low * upper_t)
upper_more_damp = upper_anchor_amp * np.exp(-beta_peak_high * upper_t)

# Envolvente inferior desde Min 2015
lower_anchor_date = troughs.iloc[0]["date"]
lower_anchor_amp = abs(troughs.iloc[0]["phase"])

lower_t = np.array([
    max(0, year_diff(d, lower_anchor_date))
    for d in env_dates
])

lower_less_damp = -lower_anchor_amp * np.exp(-beta_trough_low * lower_t)
lower_more_damp = -lower_anchor_amp * np.exp(-beta_trough_high * lower_t)

# ============================================================
# 9. Gráfico final mejorado
# ============================================================

plt.figure(figsize=(18, 8))
ax = plt.gca()

# ------------------------------------------------------------
# Fondo pastel por ciclos
# ------------------------------------------------------------

# Ciclos definidos de máximo a máximo
cycle_spans = [
    (pd.Timestamp("2013-11-01"), pd.Timestamp("2017-12-31"), "#eaf4ff", "Ciclo 2013-2017"),
    (pd.Timestamp("2017-12-31"), pd.Timestamp("2021-11-30"), "#fff3e6", "Ciclo 2017-2021"),
    (pd.Timestamp("2021-11-30"), pd.Timestamp("2025-12-31"), "#edf8ee", "Ciclo 2021-2025"),
    (pd.Timestamp("2025-12-31"), future_end, "#f5edff", "Ciclo proyectado 2025-2029")
]

for start, end, color, label in cycle_spans:
    ax.axvspan(
        start,
        end,
        color=color,
        alpha=0.45,
        zorder=0
    )

# ------------------------------------------------------------
# Oscilador con detalle real
# ------------------------------------------------------------

ax.plot(
    cycle["date"],
    cycle["cycle_smooth"],
    linewidth=2.8,
    color="#3f73b5",
    label="Ciclo BTC normalizado con datos reales",
    zorder=4
)

ax.plot(
    cycle["date"],
    cycle["cycle_raw"],
    linewidth=0.7,
    alpha=0.25,
    color="#f2a65a",
    label="Detalle semanal real",
    zorder=3
)

# Línea central
ax.axhline(
    0,
    linewidth=2.8,
    color="#3f73b5",
    alpha=0.9,
    label="Tendencia normalizada",
    zorder=2
)

# ------------------------------------------------------------
# Envolvente superior
# ------------------------------------------------------------

ax.fill_between(
    env_dates,
    upper_more_damp,
    upper_less_damp,
    color="#ff7b72",
    alpha=0.16,
    label="Envolvente superior ± amortiguamiento",
    zorder=1
)

ax.plot(
    env_dates,
    upper_less_damp,
    "--",
    color="#ff7b72",
    linewidth=2,
    zorder=5
)

ax.plot(
    env_dates,
    upper_more_damp,
    "--",
    color="#ff7b72",
    linewidth=2,
    zorder=5
)

# ------------------------------------------------------------
# Envolvente inferior
# ------------------------------------------------------------

ax.fill_between(
    env_dates,
    lower_less_damp,
    lower_more_damp,
    color="#2fbf71",
    alpha=0.16,
    label="Envolvente inferior ± amortiguamiento",
    zorder=1
)

ax.plot(
    env_dates,
    lower_less_damp,
    "--",
    color="#2fbf71",
    linewidth=2,
    zorder=5
)

ax.plot(
    env_dates,
    lower_more_damp,
    "--",
    color="#2fbf71",
    linewidth=2,
    zorder=5
)

# ------------------------------------------------------------
# Ventana próximo mínimo
# ------------------------------------------------------------

ax.axvspan(
    next_min_early,
    next_min_late,
    color="#f6bd60",
    alpha=0.22,
    label="Ventana próximo mínimo",
    zorder=1
)

ax.axvline(
    next_min_center,
    color="#f6bd60",
    linestyle=":",
    linewidth=3,
    zorder=6
)

# ------------------------------------------------------------
# Ventana próximo máximo
# ------------------------------------------------------------

ax.axvspan(
    next_max_early,
    next_max_late,
    color="#c77dff",
    alpha=0.18,
    label="Ventana próximo máximo",
    zorder=1
)

ax.axvline(
    next_max_center,
    color="#c77dff",
    linestyle=":",
    linewidth=3,
    zorder=6
)

# ------------------------------------------------------------
# Puntos clave reales
# ------------------------------------------------------------

ax.scatter(
    turns_actual["date"],
    turns_actual["phase"],
    s=70,
    color="#3f73b5",
    zorder=7
)

# Etiquetas de puntos clave
for _, row in turns_actual.iterrows():
    text = f"{row['label']}\n${row['price']:,.0f}"
    offset = 12 if row["phase"] >= 0 else -28
    va = "bottom" if row["phase"] >= 0 else "top"

    ax.annotate(
        text,
        (row["date"], row["phase"]),
        textcoords="offset points",
        xytext=(0, offset),
        ha="center",
        va=va,
        fontsize=8.5,
        zorder=8
    )

# ------------------------------------------------------------
# Punto actual
# ------------------------------------------------------------

last = weekly.iloc[-1]
last_cycle = cycle.iloc[-1]

ax.scatter(
    last_cycle["date"],
    last_cycle["cycle_smooth"],
    s=90,
    color="#f28e2b",
    zorder=8
)

ax.annotate(
    f"Hoy\n${last['price']:,.0f}",
    (last_cycle["date"], last_cycle["cycle_smooth"]),
    textcoords="offset points",
    xytext=(12, -8),
    ha="left",
    va="center",
    fontsize=9,
    zorder=9
)

# ------------------------------------------------------------
# Textos de proyección
# ------------------------------------------------------------

min_text = (
    "Próximo mínimo\n"
    f"{next_min_early.strftime('%b %Y')} – {next_min_late.strftime('%b %Y')}\n"
    f"${min_price_low:,.0f} – ${min_price_high:,.0f}"
)

max_text = (
    "Próximo máximo\n"
    f"{next_max_early.strftime('%b %Y')} – {next_max_late.strftime('%b %Y')}\n"
    f"${max_price_low:,.0f} – ${max_price_high:,.0f}"
)

ax.text(
    next_min_center,
    -0.92,
    min_text,
    ha="center",
    va="top",
    fontsize=10,
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="white",
        edgecolor="#f6bd60",
        alpha=0.96
    ),
    zorder=10
)

# Lo bajo un poco para que no choque con el borde superior
ax.text(
    next_max_center,
    0.82,
    max_text,
    ha="center",
    va="bottom",
    fontsize=10,
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="white",
        edgecolor="#c77dff",
        alpha=0.96
    ),
    zorder=10
)

# ------------------------------------------------------------
# Etiquetas visuales de cada ciclo
# ------------------------------------------------------------

cycle_labels = [
    (pd.Timestamp("2015-12-01"), "Ciclo 2013-2017"),
    (pd.Timestamp("2019-11-01"), "Ciclo 2017-2021"),
    (pd.Timestamp("2023-11-01"), "Ciclo 2021-2025"),
    (pd.Timestamp("2028-02-01"), "Ciclo proyectado")
]

for x_pos, label in cycle_labels:
    ax.text(
        x_pos,
        -1.08,
        label,
        ha="center",
        va="bottom",
        fontsize=9,
        alpha=0.65
    )

# ------------------------------------------------------------
# Eje Y estilo ciclo
# ------------------------------------------------------------

ax.set_yticks([-1.0, -0.5, 0.0, 0.5, 1.0])

ax.set_yticklabels([
    "Capitulación",
    "Infravalorado",
    "Tendencia",
    "Sobrecalentado",
    "Euforia"
])

ax.set_ylim(-1.15, 1.15)
ax.set_xlim(pd.Timestamp("2013-01-01"), future_end)

ax.set_title(
    "Bitcoin: oscilador de ciclos desde 2013 con detalle semanal real\n"
    "y envolventes exponenciales de amortiguamiento",
    fontsize=15
)

ax.set_xlabel("Fecha")
ax.set_ylabel("Fase del ciclo")
ax.grid(True, axis="y", alpha=0.3)

# ------------------------------------------------------------
# Leyenda fuera del gráfico
# ------------------------------------------------------------

ax.legend(
    loc="upper left",
    bbox_to_anchor=(1.01, 1.0),
    fontsize=9,
    frameon=True
)

# Dejamos margen a la derecha para la leyenda
plt.tight_layout(rect=[0, 0, 0.82, 1])

plt.show()

# ============================================================
# 10. Resumen numérico
# ============================================================

print("Parámetros de amortiguamiento:")
print(f"Beta superior menos amortiguamiento: {beta_peak_low:.4f} / año")
print(f"Beta superior más amortiguamiento:   {beta_peak_high:.4f} / año")
print(f"Beta inferior menos amortiguamiento: {beta_trough_low:.4f} / año")
print(f"Beta inferior más amortiguamiento:   {beta_trough_high:.4f} / año")
print()
print(f"Periodo pico→pico: {pp_mean:.2f} ± {pp_std:.2f} años")
print(f"Tiempo pico→mínimo: {pt_mean:.2f} ± {pt_std:.2f} años")
print()
print("Proyección:")
print(f"Próximo mínimo: {next_min_early.strftime('%Y-%m-%d')} a {next_min_late.strftime('%Y-%m-%d')}")
print(f"Precio mínimo estimado: ${min_price_low:,.0f} – ${min_price_high:,.0f}")
print()
print(f"Próximo máximo: {next_max_early.strftime('%Y-%m-%d')} a {next_max_late.strftime('%Y-%m-%d')}")
print(f"Precio máximo estimado: ${max_price_low:,.0f} – ${max_price_high:,.0f}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    from scipy.interpolate import PchipInterpolator
except Exception:
    PchipInterpolator = None

# ============================================================
# 0. CONFIGURACIÓN
# ============================================================

START_DATE = "2013-01-01"

N_MODELS = 3000
N_PLOT = 700
HORIZON_DAYS = 1300

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Menos suavidad = más pulso
rolling_smooth = 1

# Hipótesis de drawdown desde máximo 2025
# Esto define rango de mínimo futuro
drawdown_suave = 0.66
drawdown_duro = 0.76

# Multiplicador mínimo -> máximo siguiente
# Esto define rango de máximo futuro
mult_min = 3.0
mult_max = 6.0

# Precios de entrada que quieres estudiar
entry_prices = np.arange(25000, 70001, 2500)

# Targets para medir "me quedé fuera y se fue"
target_prices = [100000, 150000, 200000, 250000]

# ============================================================
# 1. PREPARAR DATASET
# ============================================================

df = df.rename(columns={
    "Time": "date",
    "BTC / USD Denominated Closing Price": "price"
})

df["date"] = pd.to_datetime(df["date"])
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df = df.dropna(subset=["date", "price"])
df = df.sort_values("date")

btc = df[df["date"] >= START_DATE].copy()

daily = (
    btc.set_index("date")
       .resample("D")
       .last()
       .ffill()
       .dropna()
       .reset_index()
)

daily["log_price"] = np.log(daily["price"])

current_date = daily["date"].iloc[-1]
current_price = daily["price"].iloc[-1]

print("Fecha actual del dataset:", current_date.date())
print("Precio actual:", round(current_price, 2))

# ============================================================
# 2. DETECTAR PUNTOS CLAVE REALES
# ============================================================

def get_turning_point(start, end, kind, label, phase):
    subset = daily[(daily["date"] >= start) & (daily["date"] <= end)].copy()

    if len(subset) == 0:
        raise ValueError(f"No hay datos entre {start} y {end}")

    if kind == "max":
        row = subset.loc[subset["price"].idxmax()]
    else:
        row = subset.loc[subset["price"].idxmin()]

    return {
        "date": row["date"],
        "price": row["price"],
        "log_price": np.log(row["price"]),
        "label": label,
        "phase": phase,
        "kind": kind
    }

turns_actual = []

turns_actual.append(get_turning_point("2013-01-01", "2013-04-30", "min", "Inicio 2013", -0.35))

turns_actual.append(get_turning_point("2013-10-01", "2014-01-31", "max", "Max 2013", 1.00))
turns_actual.append(get_turning_point("2014-12-01", "2015-03-31", "min", "Min 2015", -0.75))

turns_actual.append(get_turning_point("2017-11-01", "2018-01-31", "max", "Max 2017", 0.75))
turns_actual.append(get_turning_point("2018-11-01", "2019-02-28", "min", "Min 2018", -0.50))

turns_actual.append(get_turning_point("2021-10-01", "2021-12-31", "max", "Max 2021", 0.50))
turns_actual.append(get_turning_point("2022-10-01", "2023-01-31", "min", "Min 2022", -0.35))

turns_actual.append(get_turning_point("2025-01-01", "2026-03-01", "max", "Max 2025", 0.35))

turns_actual = pd.DataFrame(turns_actual).sort_values("date").reset_index(drop=True)

# ============================================================
# 3. AMORTIGUAMIENTO Y PERIODOS HISTÓRICOS
# ============================================================

def year_diff(d2, d1):
    return (pd.Timestamp(d2) - pd.Timestamp(d1)).days / 365.25

peaks = turns_actual[turns_actual["kind"] == "max"].copy()

troughs = turns_actual[
    (turns_actual["kind"] == "min") &
    (turns_actual["label"] != "Inicio 2013")
].copy()

peak_dates = peaks["date"].to_list()
peak_amps = peaks["phase"].abs().to_numpy()

trough_dates = troughs["date"].to_list()
trough_amps = troughs["phase"].abs().to_numpy()

peak_betas = []

for i in range(len(peak_amps) - 1):
    dt = year_diff(peak_dates[i + 1], peak_dates[i])
    beta = -np.log(peak_amps[i + 1] / peak_amps[i]) / dt
    peak_betas.append(beta)

trough_betas = []

for i in range(len(trough_amps) - 1):
    dt = year_diff(trough_dates[i + 1], trough_dates[i])
    beta = -np.log(trough_amps[i + 1] / trough_amps[i]) / dt
    trough_betas.append(beta)

peak_betas = np.array(peak_betas)
trough_betas = np.array(trough_betas)

beta_peak_mean = peak_betas.mean()
beta_peak_std = peak_betas.std(ddof=1)

beta_trough_mean = trough_betas.mean()
beta_trough_std = trough_betas.std(ddof=1)

beta_peak_low = max(0.001, beta_peak_mean - beta_peak_std)
beta_peak_high = beta_peak_mean + beta_peak_std

beta_trough_low = max(0.001, beta_trough_mean - beta_trough_std)
beta_trough_high = beta_trough_mean + beta_trough_std

peak_to_peak = []

for i in range(len(peak_dates) - 1):
    peak_to_peak.append(year_diff(peak_dates[i + 1], peak_dates[i]))

historical_peak_to_trough = [
    year_diff(troughs.iloc[0]["date"], peaks.iloc[0]["date"]),
    year_diff(troughs.iloc[1]["date"], peaks.iloc[1]["date"]),
    year_diff(troughs.iloc[2]["date"], peaks.iloc[2]["date"])
]

peak_to_peak = np.array(peak_to_peak)
historical_peak_to_trough = np.array(historical_peak_to_trough)

pp_mean = peak_to_peak.mean()
pp_std = peak_to_peak.std(ddof=1)

pt_mean = historical_peak_to_trough.mean()
pt_std = historical_peak_to_trough.std(ddof=1)

last_peak = peaks.iloc[-1]
last_peak_date = last_peak["date"]
last_peak_price = last_peak["price"]

next_min_early = last_peak_date + pd.to_timedelta((pt_mean - pt_std) * 365.25, unit="D")
next_min_late = last_peak_date + pd.to_timedelta((pt_mean + pt_std) * 365.25, unit="D")
next_min_center = last_peak_date + pd.to_timedelta(pt_mean * 365.25, unit="D")

next_max_early = last_peak_date + pd.to_timedelta((pp_mean - pp_std) * 365.25, unit="D")
next_max_late = last_peak_date + pd.to_timedelta((pp_mean + pp_std) * 365.25, unit="D")
next_max_center = last_peak_date + pd.to_timedelta(pp_mean * 365.25, unit="D")

# Rangos base de precio
min_price_low = last_peak_price * (1 - drawdown_duro)
min_price_high = last_peak_price * (1 - drawdown_suave)

max_price_low = min_price_low * mult_min
max_price_high = min_price_high * mult_max

print("Rango próximo mínimo:", round(min_price_low), "-", round(min_price_high))
print("Rango próximo máximo:", round(max_price_low), "-", round(max_price_high))

# ============================================================
# 4. ENVOLVENTES EXPONENCIALES
# ============================================================

future_end = current_date + pd.Timedelta(days=HORIZON_DAYS)

env_dates = pd.date_range(peaks.iloc[0]["date"], future_end, periods=900)

upper_anchor_date = peaks.iloc[0]["date"]
upper_anchor_amp = abs(peaks.iloc[0]["phase"])

lower_anchor_date = troughs.iloc[0]["date"]
lower_anchor_amp = abs(troughs.iloc[0]["phase"])

def upper_env(date, beta):
    t = max(0, year_diff(date, upper_anchor_date))
    return upper_anchor_amp * np.exp(-beta * t)

def lower_env(date, beta):
    t = max(0, year_diff(date, lower_anchor_date))
    return -lower_anchor_amp * np.exp(-beta * t)

upper_t = np.array([
    max(0, year_diff(d, upper_anchor_date))
    for d in env_dates
])

lower_t = np.array([
    max(0, year_diff(d, lower_anchor_date))
    for d in env_dates
])

upper_less_damp = upper_anchor_amp * np.exp(-beta_peak_low * upper_t)
upper_more_damp = upper_anchor_amp * np.exp(-beta_peak_high * upper_t)

lower_less_damp = -lower_anchor_amp * np.exp(-beta_trough_low * lower_t)
lower_more_damp = -lower_anchor_amp * np.exp(-beta_trough_high * lower_t)

# ============================================================
# 5. OSCILADOR HISTÓRICO DETALLADO
# ============================================================

# Proyectamos un mínimo base para mapear el tramo actual
base_min_price = (min_price_low + min_price_high) / 2
base_min_phase = np.mean([
    lower_env(next_min_center, beta_trough_low),
    lower_env(next_min_center, beta_trough_high)
])

projected_min_base = {
    "date": next_min_center,
    "price": base_min_price,
    "log_price": np.log(base_min_price),
    "label": "Min proyectado",
    "phase": base_min_phase,
    "kind": "min"
}

turns_for_mapping = pd.concat([
    turns_actual,
    pd.DataFrame([projected_min_base])
]).sort_values("date").reset_index(drop=True)

segments = []

for i in range(len(turns_for_mapping) - 1):
    start = turns_for_mapping.iloc[i]
    end = turns_for_mapping.iloc[i + 1]

    seg_end_date = min(end["date"], daily["date"].max())

    seg = daily[
        (daily["date"] >= start["date"]) &
        (daily["date"] <= seg_end_date)
    ].copy()

    if len(seg) == 0:
        continue

    log_start = start["log_price"]
    log_end = end["log_price"]
    phase_start = start["phase"]
    phase_end = end["phase"]

    if abs(log_end - log_start) < 1e-9:
        continue

    seg["ratio"] = (seg["log_price"] - log_start) / (log_end - log_start)
    seg["cycle_raw"] = phase_start + seg["ratio"] * (phase_end - phase_start)

    low = min(phase_start, phase_end) - 0.12
    high = max(phase_start, phase_end) + 0.12

    seg["cycle_raw"] = seg["cycle_raw"].clip(low, high)

    segments.append(seg)

cycle = pd.concat(segments).sort_values("date").drop_duplicates("date")

cycle["cycle_smooth"] = (
    cycle["cycle_raw"]
    .rolling(rolling_smooth, center=True, min_periods=1)
    .mean()
)

current_phase = cycle["cycle_smooth"].iloc[-1]

print("Fase actual del oscilador:", round(current_phase, 3))

# ============================================================
# 6. GENERADOR DE RUIDO CON PUENTE
# ============================================================

def bridge_noise(n, scale=0.04, rho=0.96):
    """
    Ruido AR(1) que empieza y acaba cerca de cero.
    Sirve para meter pulso sin romper los puntos clave.
    """
    if n <= 2:
        return np.zeros(n)

    eps = np.random.normal(0, scale * np.sqrt(1 - rho ** 2), n)
    z = np.zeros(n)

    for i in range(1, n):
        z[i] = rho * z[i - 1] + eps[i]

    bridge = np.linspace(z[0], z[-1], n)

    return z - bridge


def interp_curve(x_knots, y_knots, x_grid):
    x_knots = np.array(x_knots, dtype=float)
    y_knots = np.array(y_knots, dtype=float)

    order = np.argsort(x_knots)
    x_knots = x_knots[order]
    y_knots = y_knots[order]

    unique_x, unique_idx = np.unique(x_knots, return_index=True)
    unique_y = y_knots[unique_idx]

    if len(unique_x) < 2:
        return np.ones_like(x_grid) * unique_y[0]

    if PchipInterpolator is not None and len(unique_x) >= 3:
        return PchipInterpolator(unique_x, unique_y)(x_grid)
    else:
        return np.interp(x_grid, unique_x, unique_y)

# ============================================================
# 7. SIMULAR MODELOS EN ESPACIO OSCILADOR + PRECIO
# ============================================================

future_dates = pd.date_range(
    current_date + pd.Timedelta(days=1),
    periods=HORIZON_DAYS,
    freq="D"
)

x_grid = np.arange(HORIZON_DAYS)

phase_paths = []
price_paths = []
scenario_meta = []

attempts = 0

while len(phase_paths) < N_MODELS and attempts < N_MODELS * 5:
    attempts += 1

    # -------------------------
    # Muestrear fecha del mínimo
    # -------------------------
    min_delay_years = np.random.normal(pt_mean, pt_std)
    min_delay_years = np.clip(min_delay_years, pt_mean - 2 * pt_std, pt_mean + 2 * pt_std)

    min_date = last_peak_date + pd.to_timedelta(min_delay_years * 365.25, unit="D")

    if min_date <= current_date + pd.Timedelta(days=20):
        min_date = current_date + pd.Timedelta(days=np.random.randint(40, 180))

    # -------------------------
    # Muestrear fecha del máximo
    # -------------------------
    max_delay_years = np.random.normal(pp_mean, pp_std)
    max_delay_years = np.clip(max_delay_years, pp_mean - 2 * pp_std, pp_mean + 2 * pp_std)

    max_date = last_peak_date + pd.to_timedelta(max_delay_years * 365.25, unit="D")

    if max_date <= min_date + pd.Timedelta(days=365):
        max_date = min_date + pd.Timedelta(days=np.random.randint(700, 1200))

    # -------------------------
    # Muestrear precios
    # -------------------------
    dd = np.random.normal(0.71, 0.05)
    dd = np.clip(dd, 0.55, 0.85)

    min_price = last_peak_price * (1 - dd)

    mult = np.random.normal(4.2, 1.1)
    mult = np.clip(mult, 2.2, 7.5)

    max_price = min_price * mult

    # Filtro de cordura
    if min_price < 15000 or min_price > current_price * 0.95:
        continue

    if max_price < current_price * 1.15:
        continue

    if max_price > 450000:
        continue

    # -------------------------
    # Muestrear amortiguamientos
    # -------------------------
    beta_p = np.random.uniform(beta_peak_low, beta_peak_high)
    beta_t = np.random.uniform(beta_trough_low, beta_trough_high)

    max_phase = upper_env(max_date, beta_p)
    min_phase = lower_env(min_date, beta_t)

    # Añadimos un poco de variación
    max_phase *= np.random.uniform(0.90, 1.10)
    min_phase *= np.random.uniform(0.90, 1.10)

    max_phase = np.clip(max_phase, 0.10, 0.45)
    min_phase = np.clip(min_phase, -0.50, -0.10)

    # -------------------------
    # Índices temporales
    # -------------------------
    min_idx = int((min_date - future_dates[0]).days)
    max_idx = int((max_date - future_dates[0]).days)

    min_idx = np.clip(min_idx, 5, HORIZON_DAYS - 10)
    max_idx = np.clip(max_idx, min_idx + 30, HORIZON_DAYS - 1)

    # Punto terminal después del máximo, si hay espacio
    terminal_phase = max_phase - np.random.uniform(0.00, 0.20)
    terminal_phase = np.clip(terminal_phase, -0.05, max_phase)

    terminal_price = max_price * np.exp(np.random.normal(0, 0.12))
    terminal_price = np.clip(terminal_price, max_price * 0.70, max_price * 1.25)

    x_knots = [0, min_idx, max_idx, HORIZON_DAYS - 1]

    phase_knots = [
        current_phase,
        min_phase,
        max_phase,
        terminal_phase
    ]

    log_price_knots = [
        np.log(current_price),
        np.log(min_price),
        np.log(max_price),
        np.log(terminal_price)
    ]

    # -------------------------
    # Curva base
    # -------------------------
    phase_base = interp_curve(x_knots, phase_knots, x_grid)
    log_price_base = interp_curve(x_knots, log_price_knots, x_grid)

    # -------------------------
    # Ruido con pulso
    # -------------------------
    phase_noise_scale = np.random.uniform(0.015, 0.060)
    price_noise_scale = np.random.uniform(0.015, 0.070)

    phase_noise = np.zeros(HORIZON_DAYS)
    price_noise = np.zeros(HORIZON_DAYS)

    knots_sorted = sorted(set(x_knots))

    for a, b in zip(knots_sorted[:-1], knots_sorted[1:]):
        n = b - a + 1

        phase_noise[a:b+1] += bridge_noise(
            n,
            scale=phase_noise_scale,
            rho=np.random.uniform(0.92, 0.985)
        )

        price_noise[a:b+1] += bridge_noise(
            n,
            scale=price_noise_scale,
            rho=np.random.uniform(0.92, 0.985)
        )

    phase_path = phase_base + phase_noise

    # Límite suave por envolventes
    upper_limit = np.array([
        upper_env(d, beta_peak_low) + 0.08
        for d in future_dates
    ])

    lower_limit = np.array([
        lower_env(d, beta_trough_low) - 0.08
        for d in future_dates
    ])

    phase_path = np.clip(phase_path, lower_limit, upper_limit)
    phase_path = np.clip(phase_path, -1.05, 1.05)

    price_path = np.exp(log_price_base + price_noise)

    # Filtro final de cordura
    if np.nanmin(price_path) < 10000:
        continue

    if np.nanmax(price_path) > 500000:
        continue

    phase_paths.append(phase_path)
    price_paths.append(price_path)

    scenario_meta.append({
        "min_date": future_dates[min_idx],
        "min_price": min_price,
        "max_date": future_dates[max_idx],
        "max_price": max_price,
        "min_phase": min_phase,
        "max_phase": max_phase,
        "drawdown": dd,
        "mult": mult
    })

phase_paths = np.array(phase_paths)
price_paths = np.array(price_paths)
scenario_meta = pd.DataFrame(scenario_meta)

print("Modelos generados:", len(phase_paths))
display(scenario_meta.head())

In [ ]:
# ============================================================
# 8. GRAFICAR FUTUROS EN EL OSCILADOR
# ============================================================

plt.figure(figsize=(18, 8))
ax = plt.gca()

# Fondos pastel por ciclo
cycle_spans = [
    (pd.Timestamp("2013-11-01"), pd.Timestamp("2017-12-31"), "#eaf4ff"),
    (pd.Timestamp("2017-12-31"), pd.Timestamp("2021-11-30"), "#fff3e6"),
    (pd.Timestamp("2021-11-30"), pd.Timestamp("2025-12-31"), "#edf8ee"),
    (pd.Timestamp("2025-12-31"), future_end, "#f5edff")
]

for start, end, color in cycle_spans:
    ax.axvspan(start, end, color=color, alpha=0.45, zorder=0)

# Histórico
ax.plot(
    cycle["date"],
    cycle["cycle_smooth"],
    linewidth=2.8,
    color="#3f73b5",
    label="Oscilador histórico",
    zorder=5
)

ax.plot(
    cycle["date"],
    cycle["cycle_raw"],
    linewidth=0.6,
    alpha=0.20,
    color="#f2a65a",
    label="Detalle real",
    zorder=4
)

ax.axhline(
    0,
    linewidth=2.8,
    color="#3f73b5",
    alpha=0.9,
    label="Tendencia",
    zorder=3
)

# Modelos futuros
n_plot = min(N_PLOT, len(phase_paths))
plot_idx = np.random.choice(np.arange(len(phase_paths)), size=n_plot, replace=False)

for idx in plot_idx:
    ax.plot(
        future_dates,
        phase_paths[idx],
        color="#6f79ff",
        alpha=0.035,
        linewidth=0.9,
        zorder=2
    )

# Bandas de escenarios
p05 = np.percentile(phase_paths, 5, axis=0)
p25 = np.percentile(phase_paths, 25, axis=0)
p50 = np.percentile(phase_paths, 50, axis=0)
p75 = np.percentile(phase_paths, 75, axis=0)
p95 = np.percentile(phase_paths, 95, axis=0)

ax.fill_between(
    future_dates,
    p05,
    p95,
    color="#6f79ff",
    alpha=0.10,
    label="Escenarios 5%-95%",
    zorder=1
)

ax.fill_between(
    future_dates,
    p25,
    p75,
    color="#6f79ff",
    alpha=0.18,
    label="Escenarios 25%-75%",
    zorder=1
)

ax.plot(
    future_dates,
    p50,
    color="#d62728",
    linewidth=2.5,
    label="Mediana escenarios",
    zorder=6
)

# Envolventes
ax.fill_between(
    env_dates,
    upper_more_damp,
    upper_less_damp,
    color="#ff7b72",
    alpha=0.14,
    label="Envolvente superior",
    zorder=1
)

ax.plot(env_dates, upper_less_damp, "--", color="#ff7b72", linewidth=2, zorder=6)
ax.plot(env_dates, upper_more_damp, "--", color="#ff7b72", linewidth=2, zorder=6)

ax.fill_between(
    env_dates,
    lower_less_damp,
    lower_more_damp,
    color="#2fbf71",
    alpha=0.14,
    label="Envolvente inferior",
    zorder=1
)

ax.plot(env_dates, lower_less_damp, "--", color="#2fbf71", linewidth=2, zorder=6)
ax.plot(env_dates, lower_more_damp, "--", color="#2fbf71", linewidth=2, zorder=6)

# Ventanas de mínimo y máximo
ax.axvspan(next_min_early, next_min_late, color="#f6bd60", alpha=0.18, label="Ventana mínimo")
ax.axvline(next_min_center, color="#f6bd60", linestyle=":", linewidth=3)

ax.axvspan(next_max_early, next_max_late, color="#c77dff", alpha=0.16, label="Ventana máximo")
ax.axvline(next_max_center, color="#c77dff", linestyle=":", linewidth=3)

# Puntos históricos
ax.scatter(
    turns_actual["date"],
    turns_actual["phase"],
    s=70,
    color="#3f73b5",
    zorder=7
)

for _, row in turns_actual.iterrows():
    text = f"{row['label']}\n${row['price']:,.0f}"
    offset = 12 if row["phase"] >= 0 else -28
    va = "bottom" if row["phase"] >= 0 else "top"

    ax.annotate(
        text,
        (row["date"], row["phase"]),
        textcoords="offset points",
        xytext=(0, offset),
        ha="center",
        va=va,
        fontsize=8.5,
        zorder=8
    )

# Punto actual
ax.scatter(
    current_date,
    current_phase,
    s=100,
    color="#f28e2b",
    zorder=9
)

ax.annotate(
    f"Hoy\n${current_price:,.0f}",
    (current_date, current_phase),
    textcoords="offset points",
    xytext=(12, -8),
    ha="left",
    va="center",
    fontsize=9,
    zorder=10
)

# Textos de proyección
min_text = (
    "Zona probable de mínimo\n"
    f"{next_min_early.strftime('%b %Y')} – {next_min_late.strftime('%b %Y')}\n"
    f"${min_price_low:,.0f} – ${min_price_high:,.0f}"
)

max_text = (
    "Zona probable de máximo\n"
    f"{next_max_early.strftime('%b %Y')} – {next_max_late.strftime('%b %Y')}\n"
    f"${max_price_low:,.0f} – ${max_price_high:,.0f}"
)

ax.text(
    next_min_center,
    -0.92,
    min_text,
    ha="center",
    va="top",
    fontsize=10,
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="white",
        edgecolor="#f6bd60",
        alpha=0.96
    ),
    zorder=10
)

ax.text(
    next_max_center,
    0.82,
    max_text,
    ha="center",
    va="bottom",
    fontsize=10,
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="white",
        edgecolor="#c77dff",
        alpha=0.96
    ),
    zorder=10
)

ax.set_yticks([-1.0, -0.5, 0.0, 0.5, 1.0])

ax.set_yticklabels([
    "Capitulación",
    "Infravalorado",
    "Tendencia",
    "Sobrecalentado",
    "Euforia"
])

ax.set_ylim(-1.15, 1.15)
ax.set_xlim(pd.Timestamp("2013-01-01"), future_end)

ax.set_title(
    "Bitcoin: ensemble de futuros simulados dentro del oscilador de ciclo",
    fontsize=15
)

ax.set_xlabel("Fecha")
ax.set_ylabel("Fase del ciclo")
ax.grid(True, axis="y", alpha=0.3)

ax.legend(
    loc="upper left",
    bbox_to_anchor=(1.01, 1.0),
    fontsize=9,
    frameon=True
)

plt.tight_layout(rect=[0, 0, 0.82, 1])
plt.show()

In [ ]:
# ============================================================
# 9. PROBABILIDADES SEGÚN PRECIO DE COMPRA
# ============================================================

def analyze_entry_prices(price_paths, future_dates, current_price, entry_prices, target_prices):
    results = []

    terminal_prices = price_paths[:, -1]
    max_prices = price_paths.max(axis=1)

    for entry in entry_prices:
        touched = (price_paths <= entry).any(axis=1)
        p_touch = touched.mean()
        p_miss = 1 - p_touch

        days_to_touch = []
        after_drop_20 = []
        after_drop_30 = []
        after_double = []

        wait_terminal_mult = []
        buy_now_terminal_mult = terminal_prices / current_price

        target_before_entry_probs = {}

        for target in target_prices:
            target_before_entry = []

            for i in range(price_paths.shape[0]):
                path = price_paths[i]

                entry_hits = np.where(path <= entry)[0]
                target_hits = np.where(path >= target)[0]

                if len(target_hits) == 0:
                    target_before_entry.append(False)
                else:
                    first_target = target_hits[0]

                    if len(entry_hits) == 0:
                        target_before_entry.append(True)
                    else:
                        first_entry = entry_hits[0]
                        target_before_entry.append(first_target < first_entry)

            target_before_entry_probs[target] = np.mean(target_before_entry)

        for i in range(price_paths.shape[0]):
            path = price_paths[i]
            hit_idx = np.where(path <= entry)[0]

            if len(hit_idx) > 0:
                first_hit = hit_idx[0]
                days_to_touch.append(first_hit + 1)

                post = path[first_hit:]

                after_drop_20.append(post.min() <= entry * 0.80)
                after_drop_30.append(post.min() <= entry * 0.70)
                after_double.append(post.max() >= entry * 2.00)

                wait_terminal_mult.append(path[-1] / entry)
            else:
                wait_terminal_mult.append(1.0)

        if len(days_to_touch) > 0:
            median_days = np.median(days_to_touch)
            mean_days = np.mean(days_to_touch)
            p_drop20 = np.mean(after_drop_20)
            p_drop30 = np.mean(after_drop_30)
            p_double = np.mean(after_double)
        else:
            median_days = np.nan
            mean_days = np.nan
            p_drop20 = np.nan
            p_drop30 = np.nan
            p_double = np.nan

        row = {
            "entry_price": entry,
            "p_toca_entry": p_touch,
            "p_quedarse_fuera": p_miss,
            "dias_medianos_hasta_compra": median_days,
            "dias_medios_hasta_compra": mean_days,
            "p_caida_20_despues_de_comprar": p_drop20,
            "p_caida_30_despues_de_comprar": p_drop30,
            "p_doblar_despues_de_comprar": p_double,
            "valor_esperado_esperar": np.mean(wait_terminal_mult),
            "valor_mediano_esperar": np.median(wait_terminal_mult),
            "valor_esperado_comprar_hoy": np.mean(buy_now_terminal_mult),
            "valor_mediano_comprar_hoy": np.median(buy_now_terminal_mult)
        }

        for target in target_prices:
            row[f"p_llega_{target}_antes_de_entry"] = target_before_entry_probs[target]

        results.append(row)

    return pd.DataFrame(results)


entry_analysis = analyze_entry_prices(
    price_paths=price_paths,
    future_dates=future_dates,
    current_price=current_price,
    entry_prices=entry_prices,
    target_prices=target_prices
)

display_df = entry_analysis.copy()

percent_cols = [
    "p_toca_entry",
    "p_quedarse_fuera",
    "p_caida_20_despues_de_comprar",
    "p_caida_30_despues_de_comprar",
    "p_doblar_despues_de_comprar"
]

for target in target_prices:
    percent_cols.append(f"p_llega_{target}_antes_de_entry")

for col in percent_cols:
    display_df[col] = (display_df[col] * 100).round(1)

value_cols = [
    "valor_esperado_esperar",
    "valor_mediano_esperar",
    "valor_esperado_comprar_hoy",
    "valor_mediano_comprar_hoy"
]

for col in value_cols:
    display_df[col] = display_df[col].round(2)

display_df["dias_medianos_hasta_compra"] = display_df["dias_medianos_hasta_compra"].round(0)
display_df["dias_medios_hasta_compra"] = display_df["dias_medios_hasta_compra"].round(0)

display(display_df)

In [ ]:
# ============================================================
# 10. GRÁFICO DE RIESGO DE QUEDARSE FUERA
# ============================================================

plt.figure(figsize=(14, 6))
ax = plt.gca()

ax.plot(
    entry_analysis["entry_price"],
    entry_analysis["p_quedarse_fuera"] * 100,
    marker="o",
    linewidth=2.5,
    label="P(quedarse fuera)"
)

ax.plot(
    entry_analysis["entry_price"],
    entry_analysis["p_toca_entry"] * 100,
    marker="o",
    linewidth=2.5,
    label="P(toca precio de compra)"
)

ax.plot(
    entry_analysis["entry_price"],
    entry_analysis["p_llega_150000_antes_de_entry"] * 100,
    marker="o",
    linewidth=2.5,
    label="P(llega a 150k antes de tu entrada)"
)

ax.axvline(
    current_price,
    linestyle="--",
    color="gray",
    label="Precio actual"
)

ax.set_title("Probabilidad de quedarse fuera según precio de compra")
ax.set_xlabel("Precio límite de compra BTC")
ax.set_ylabel("Probabilidad (%)")
ax.grid(True, alpha=0.3)
ax.legend()

plt.show()